In [1]:
import heapq
import random
from bisect import bisect_left, insort
from collections import Counter, defaultdict, deque

# 1) Greedy

**The gist.** Make the choice that looks best right now and never take it back. It's only correct if you can argue that some best answer makes the same first choice.

> **Think of it like…** paying with the fewest banknotes: hand over the biggest note that fits, repeat. It works for euros because of how the notes are sized — and fails for weird coin systems, which is why greedy needs a reason.

**The exchange argument.** Take a best solution that differs from yours at the first choice; show you can swap in your choice without making it worse. Can't find that argument in a minute? It's probably DP.

**Spot it:** "minimum number of jumps / arrows / boats", "maximum number of…", a natural sort order (deadline, end time, difference), one running quantity (reach, tank, balance), constraints from both sides (two passes).

**Classic bugs:** greedy where a counterexample exists (coins `[1, 3, 4]`) · sorting by the wrong key · forgetting the global check (Gas Station total, Hand of Straights length).

## 1.1 Reachability and scans

Each question keeps one or two running quantities (reach, tank, balance) and updates them in one pass. The real work is explaining why the greedy step is safe.

### 1.1.1 [55. Jump Game](https://leetcode.com/problems/jump-game/) — ★★★ · Medium

**Problem.** `nums[i]` is the max jump from `i`; can you reach the last index from index 0?  
**Example.** `[2, 3, 1, 1, 4]` → `True`; `[3, 2, 1, 0, 4]` → `False` (you get stuck on the 0)  
**Spot it.** Track the farthest reachable index.

**Ask first**
- Can values be 0? → Yes — a 0 is exactly how you get stuck.

> 💡 **Idea.** Walk left to right tracking the farthest index you can reach. If you ever stand on an index beyond that, you're stuck.

**Brute force:** DP: mark each index reachable from earlier ones, O(n²).

**Steps**
1. `reach = 0`.
2. For each `i`, if `i > reach`, return `False`.
3. Otherwise `reach = max(reach, i + nums[i])`.
4. Finishing the loop → `True`.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **Fewest jumps to the end (LC 45)?** Greedy BFS levels: count a jump each time you pass the end of the current reach window.
- **Return a path, not just yes/no?** Store, for each index, the earlier index that first reached it; walk back from the end.
- **You can jump left or right (LC 1306)?** Now it's a graph — BFS/DFS from the start with a visited set.

In [2]:
def can_jump(nums):
    """True if the last index is reachable."""
    # every index up to reach can be reached
    reach = 0
    for i, step in enumerate(nums):
        # a gap: index i can never be reached
        if i > reach:
            return False
        reach = max(reach, i + step)
    return True


assert can_jump([2, 3, 1, 1, 4])
assert not can_jump([3, 2, 1, 0, 4])
print(can_jump([2, 3, 1, 1, 4]))

True


### 1.1.2 [45. Jump Game II](https://leetcode.com/problems/jump-game-ii/) — ★★ · Medium

**Problem.** Minimum number of jumps to reach the last index (always reachable).  
**Example.** `[2, 3, 1, 1, 4]` → `2` (jump 1 step, then 3 steps)  
**Spot it.** Implicit BFS by levels: each jump extends the window of reachable indices.

**Ask first**
- Is the end always reachable? → Yes here. If not, return −1 when the window stops growing.

> 💡 **Idea.** Think BFS levels: everything reachable with k jumps forms a window. Scan it, track the farthest next reach; when you hit the window's end, that's one more jump.

**Brute force:** DP: `jumps[j] = min(jumps[i] + 1)` for all i that reach j, O(n²).

**Steps**
1. `end` = edge of the current jump's window; `far` = farthest reachable from it.
2. For each `i` (except the last), update `far`.
3. When `i == end`, take a jump: `jumps += 1`, `end = far`.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **Return the actual jumps?** In each window remember which index gave the farthest reach; those indices are the path (e.g. `[2,3,1,1,4]` → 0 → 1 → 4).
- **End might be unreachable?** If the farthest reach stops growing before the end, return −1.
- **Each jump has a cost?** Dijkstra or DP over indices.

In [3]:
def jump(nums):
    """Fewest jumps to reach the end."""
    jumps = end = far = 0
    # the last index never needs a jump from it
    for i in range(len(nums) - 1):
        far = max(far, i + nums[i])
        # leaving the current window costs one more jump
        if i == end:
            jumps += 1
            end = far
    return jumps


assert jump([2, 3, 1, 1, 4]) == 2
assert jump([2, 3, 0, 1, 4]) == 2
assert jump([0]) == 0
print(jump([2, 3, 1, 1, 4]))

2


### 1.1.3 [134. Gas Station](https://leetcode.com/problems/gas-station/) — ★★ · Medium

**Problem.** Circular route; `gas[i]` fuel at station `i`, `cost[i]` to drive to the next. Starting index for a full loop, or −1.  
**Example.** `gas = [1,2,3,4,5], cost = [3,4,5,1,2]` → `3`  
**Spot it.** If total gas ≥ total cost a start exists; a failed run rules out every start inside it.

**Ask first**
- Is the answer unique? → Yes, guaranteed if one exists.

> 💡 **Idea.** If total gas < total cost, it's impossible. Otherwise walk around; whenever the tank goes negative, no station so far can be the start — restart from the next one.

**Brute force:** try every start and simulate the full loop, O(n²).

**Steps**
1. If total gas < total cost, return −1.
2. Walk once with `tank += gas − cost`.
3. When `tank < 0`, set `start = i + 1` and reset the tank.
4. Return `start`.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **Why can you skip all stations before the failure point?** If you start at A and fail at B, starting anywhere between A and B arrives with even less gas, so they fail too.
- **Why does total gas ≥ total cost guarantee an answer?** The start right after the lowest point of the running balance never dips below zero.
- **Tank has a capacity limit?** The simple greedy breaks; simulate with the cap or use a different argument.

In [4]:
def can_complete_circuit(gas, cost):
    """Start index for a full loop, or -1."""
    # not enough fuel in total: no start can work
    if sum(gas) < sum(cost):
        return -1
    start = tank = 0
    for i, (g, c) in enumerate(zip(gas, cost)):
        tank += g - c
        # every start from `start` to i fails here: try i + 1
        if tank < 0:
            start, tank = i + 1, 0
    return start


assert can_complete_circuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]) == 3
assert can_complete_circuit([2, 3, 4], [3, 4, 3]) == -1
print(can_complete_circuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]))

3


### 1.1.4 [763. Partition Labels](https://leetcode.com/problems/partition-labels/) — ★★ · Medium

**Problem.** Split `s` into as many parts as possible so each letter appears in at most one part; return part sizes.  
**Example.** `"ababcbacadefegdehijhklij"` → `[9, 7, 8]`  
**Spot it.** A part must extend to the last occurrence of every letter inside it.

**Ask first**
- Return sizes or the substrings? → Sizes.

> 💡 **Idea.** Record each letter's last position. Walk the string stretching the current part's end to the last position of every letter you see; when you reach that end, close the part.

**Brute force:** for each cut point, check no letter appears on both sides, O(n²).

**Steps**
1. Record each letter's last index.
2. Scan, setting `end = max(end, last[ch])`.
3. When `i == end`, record the part size and start a new part.

**Cost:** Time O(n) · Space O(26)

**Likely follow-ups**
- **Return the parts themselves?** Slice using the recorded cut points.
- **String arrives as a stream?** You can't close a part until you know a letter won't appear again — needs the whole string (or an end marker).
- **Same idea on intervals?** Each letter is an interval `[first, last]`; merge overlapping intervals (LC 56).

In [5]:
def partition_labels(s):
    """Sizes of maximal-count partitions."""
    last = {ch: i for i, ch in enumerate(s)}
    out, start, end = [], 0, 0
    for i, ch in enumerate(s):
        # the part must include ch's last occurrence
        end = max(end, last[ch])
        # every letter seen so far ends here: close the part
        if i == end:
            out.append(end - start + 1)
            start = i + 1
    return out


assert partition_labels("ababcbacadefegdehijhklij") == [9, 7, 8]
assert partition_labels("eccbbbbdec") == [10]
print(partition_labels("ababcbacadefegdehijhklij"))

[9, 7, 8]


### 1.1.5 [678. Valid Parenthesis String](https://leetcode.com/problems/valid-parenthesis-string/) — ★★ · Medium

**Problem.** `(`, `)` and `*` (which can be `(`, `)` or empty); is the string valid?  
**Example.** `"(*)"` → `True`; `"(*))"` → `True`  
**Spot it.** Track the *range* of possible open counts.

**Ask first**
- Can `*` be `(`, `)` or empty? → Yes, any of the three.

> 💡 **Idea.** Track the range `[lo, hi]` of how many `(` could be open. `(` → both +1. `)` → both −1. `*` → `lo − 1`, `hi + 1`. If `hi < 0`, fail; clamp `lo` at 0. Valid if `lo == 0` at the end.

**Brute force:** try all 3^(stars) replacements, exponential.

**Steps**
1. `lo`, `hi` = min and max open count.
2. `(` → both +1; `)` → both −1; `*` → `lo − 1`, `hi + 1`.
3. `hi < 0` → invalid; clamp `lo` at 0.
4. Valid iff `lo == 0` at the end.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **Two-stack version?** Stack of `(` indices and stack of `*` indices; match `)` with `(` first, then `*`; finally each leftover `(` needs a later `*`.
- **Return one valid string?** Use the two-stack matching and turn unused `*`s into empty.
- **Several bracket types with wildcards?** The range trick no longer works — DP over `(i, stack state)` or backtracking.

In [6]:
def check_valid_string(s):
    """True if '*' can be assigned to balance s."""
    # range of possible unmatched "(" counts
    lo = hi = 0
    for ch in s:
        lo += 1 if ch == "(" else -1
        hi += -1 if ch == ")" else 1
        # even treating every star as "(" is not enough
        if hi < 0:
            return False
        # open count can't go below zero
        lo = max(lo, 0)
    return lo == 0


assert check_valid_string("()")
assert check_valid_string("(*)")
assert check_valid_string("(*))")
assert not check_valid_string("(((*)")
print(check_valid_string("(*))"))

True


### 1.1.6 [846. Hand of Straights](https://leetcode.com/problems/hand-of-straights/) — ★★ · Medium

**Problem.** Can the cards be split into groups of `group_size` consecutive values?  
**Example.** `hand = [1,2,3,6,2,3,4,7,8], group_size = 3` → `True` (`[1,2,3]`, `[2,3,4]`, `[6,7,8]`)  
**Spot it.** The smallest remaining card must start a group.

**Ask first**
- Card count not divisible by group size? → Return False right away.

> 💡 **Idea.** The smallest card left must start a group. Take it and the next `group_size − 1` values; if any is missing, fail. Repeat.

**Brute force:** try every way to group the cards, exponential.

**Steps**
1. Total must divide by the group size.
2. Count cards and visit values in sorted order.
3. If value `x` has `need` copies left, each must start a group: subtract `need` from `x, x+1, …`.
4. Any shortage → `False`.

**Cost:** Time O(n log n) · Space O(n)

**Likely follow-ups**
- **Faster than O(n log n)?** Count map + for each card, walk down to the start of its run (`x − 1` missing), then deal groups from there — O(n) average.
- **Same as LC 1296?** Yes, identical problem.
- **Split into runs of length ≥ 3 instead (LC 659)?** Greedy with "runs ending at x" counts: extend an existing run before starting a new one.

In [7]:
def is_n_straight_hand(hand, group_size):
    """True if hand splits into consecutive groups."""
    if len(hand) % group_size:
        return False
    count = Counter(hand)
    for x in sorted(count):
        # remaining copies of the smallest value must each start a group
        need = count[x]
        if need:
            for v in range(x, x + group_size):
                if count[v] < need:
                    return False
                count[v] -= need
    return True


assert is_n_straight_hand([1, 2, 3, 6, 2, 3, 4, 7, 8], 3)
assert not is_n_straight_hand([1, 2, 3, 4, 5], 4)
print(is_n_straight_hand([1, 2, 3, 6, 2, 3, 4, 7, 8], 3))

True


### 1.1.7 [1899. Merge Triplets to Form Target Triplet](https://leetcode.com/problems/merge-triplets-to-form-target-triplet/) — ★★ · Medium

**Problem.** Merging two triplets takes the element-wise max. Can you obtain `target`?  
**Example.** `triplets = [[2,5,3],[1,8,4],[1,7,5]], target = [2, 7, 5]` → `True`  
**Spot it.** Only triplets that never exceed the target in any coordinate are usable.

**Ask first**
- Merge = element-wise max? → Yes, so values can only go up.

> 💡 **Idea.** Ignore any triplet with a value bigger than the target (it would ruin that slot). From the rest, check that every target value is hit by some triplet.

**Brute force:** try every subset of triplets, O(2ⁿ).

**Steps**
1. Skip triplets with any component above the target.
2. For each safe triplet, note which coordinates equal the target.
3. Return whether all three coordinates were hit.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **Return which triplets to merge?** Every "safe" triplet that matches at least one target value.
- **k-length tuples instead of triplets?** Same idea per coordinate, O(n·k).

In [8]:
def merge_triplets(triplets, target):
    """True if target is reachable by max-merges."""
    # target coordinates matched by some safe triplet
    hit = set()
    for t in triplets:
        # a component above the target could never be undone
        if all(a <= b for a, b in zip(t, target)):
            hit.update(i for i in range(3) if t[i] == target[i])
    return len(hit) == 3


assert merge_triplets([[2, 5, 3], [1, 8, 4], [1, 7, 5]], [2, 7, 5])
assert not merge_triplets([[3, 4, 5], [4, 5, 6]], [3, 2, 5])
print(merge_triplets([[2, 5, 3], [1, 8, 4], [1, 7, 5]], [2, 7, 5]))

True


### 1.1.8 [135. Candy](https://leetcode.com/problems/candy/) — ★★ · Hard

**Problem.** Each child gets ≥ 1 candy; a child with a higher rating than a neighbour gets more than that neighbour. Minimum total?  
**Example.** `[1, 0, 2]` → `5` (candies 2, 1, 2)  
**Spot it.** Two passes: satisfy left neighbours, then right neighbours.

**Ask first**
- Equal ratings — any rule? → No, equal neighbours can get any amount.

> 💡 **Idea.** Two passes: left to right, give one more than the left neighbour if rated higher; right to left, make sure each child also beats the right neighbour (take the max).

**Brute force:** keep fixing violations until stable, O(n²).

**Steps**
1. Everyone starts with 1.
2. Left to right: if rating rises, `candy[i] = candy[i − 1] + 1`.
3. Right to left: if rating rises leftward, `candy[i] = max(candy[i], candy[i + 1] + 1)`.
4. Sum.

**Cost:** Time O(n) · Space O(n)

**Likely follow-ups**
- **O(1) extra space?** Count lengths of rising and falling slopes and add triangle numbers, adjusting the peak.

In [9]:
def candy(ratings):
    """Minimum candies satisfying neighbour constraints."""
    n = len(ratings)
    out = [1] * n
    # satisfy every left neighbour
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            out[i] = out[i - 1] + 1
    # satisfy every right neighbour without breaking the left side
    for i in range(n - 2, -1, -1):
        if ratings[i] > ratings[i + 1]:
            out[i] = max(out[i], out[i + 1] + 1)
    return sum(out)


assert candy([1, 0, 2]) == 5
assert candy([1, 2, 2]) == 4
print(candy([1, 0, 2]))

5


### 1.1.9 [406. Queue Reconstruction by Height](https://leetcode.com/problems/queue-reconstruction-by-height/) — ★★ · Medium

**Problem.** People `[h, k]`: `k` people in front are at least as tall. Reconstruct the queue.  
**Example.** `[[7,0],[4,4],[7,1],[5,0],[6,1],[5,2]]` → `[[5,0],[7,0],[5,2],[6,1],[4,4],[7,1]]`  
**Spot it.** Insert tallest first; shorter people do not affect taller people's `k`.

**Ask first**
- Does `k` count people taller *or equal*? → Yes, `>=`.

> 💡 **Idea.** Place tall people first: sort by height descending (and `k` ascending), then insert each person at index `k`. Shorter people inserted later never change what taller people see.

**Brute force:** try every order, factorial.

**Steps**
1. Sort by height descending, then `k` ascending.
2. Insert each person at index `k` of the output.
3. Return the output.

**Cost:** Time O(n²) (list insertions) · Space O(n)

**Likely follow-ups**
- **O(n log n)?** Process shortest first and put each into the k-th *empty* slot (counting only slots for people ≥ them) using a Fenwick tree to find that slot.
- **Why tallest first?** When a person is inserted, everyone already placed is taller or equal, so `k` is exactly their index.

In [10]:
def reconstruct_queue(people):
    """Queue consistent with every [h, k]."""
    out = []
    # tallest first; later (shorter) people never change earlier k values
    for p in sorted(people, key=lambda p: (-p[0], p[1])):
        out.insert(p[1], p)
    return out


people = [[7, 0], [4, 4], [7, 1], [5, 0], [6, 1], [5, 2]]
assert reconstruct_queue(people) == [[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]
print(reconstruct_queue(people))

[[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]


### 1.1.10 [334. Increasing Triplet Subsequence](https://leetcode.com/problems/increasing-triplet-subsequence/) — ★★ · Medium

**Problem.** Is there `i < j < k` with `nums[i] < nums[j] < nums[k]`?  
**Example.** `[2, 1, 5, 0, 4, 6]` → `True` (for example 1, 4, 6)  
**Spot it.** Increasing triple in O(n) time and O(1) space → keep the two smallest "ends" seen so far.

**Ask first**
- Strictly increasing? → Yes, use `<=` when updating so equal values don't count.

> 💡 **Idea.** Keep the smallest value seen (`first`) and the smallest value that has something smaller before it (`second`). Any number bigger than `second` completes a triplet.

**Brute force:** three nested loops, O(n³).

**Steps**
1. Set `first = second = ∞`.
2. For each `x`: if `x ≤ first`, set `first = x`; else if `x ≤ second`, set `second = x`; else return `True`.
3. Return `False`.

**Cost:** Time O(n) (one pass) · Space O(1) (two variables)

**Likely follow-ups**
- **Return the indices?** Also remember the index of the `first` that was current when `second` was set.
- **Increasing subsequence of length k?** Keep k − 1 smallest tails (the LIS `tails` array), stop when it reaches length k: O(n log k).
- **Why can `first` change after `second` is set?** `second` still proves *some* smaller value came before it — the triplet is valid.

In [11]:
def increasing_triplet(nums):
    """True if some i < j < k has nums[i] < nums[j] < nums[k]."""
    first = second = float("inf")
    for x in nums:
        # a new smallest start
        if x <= first:
            first = x
        # the smallest middle that has a smaller value before it
        elif x <= second:
            second = x
        # bigger than a valid middle: triple found
        else:
            return True
    return False


assert increasing_triplet([1, 2, 3, 4, 5])
assert not increasing_triplet([5, 4, 3, 2, 1])
assert increasing_triplet([2, 1, 5, 0, 4, 6])
assert not increasing_triplet([1, 1, 1])
assert increasing_triplet([20, 100, 10, 12, 5, 13])
print(increasing_triplet([2, 1, 5, 0, 4, 6]))

True


### 1.1.11 [649. Dota2 Senate](https://leetcode.com/problems/dota2-senate/) — ★★ · Medium

**Problem.** Senators of parties `R` and `D` act in order, round after round; on its turn a senator bans one opponent. Which party wins when everyone plays optimally?  
**Example.** `"RDD"` → `"Dire"`  
**Spot it.** Turn order repeats and the best move is "ban the next opponent" → two queues of turn indices.

**Ask first**
- Ban the *next* opposing senator? → Yes — that's the best move.

> 💡 **Idea.** Two queues of senator indices. The earlier one in line bans the other; the winner goes back to the queue with index `+ n` (next round). The party with senators left wins.

**Brute force:** simulate rounds with a list, removing banned senators, O(n²).

**Steps**
1. Put the indices of `R` senators in one queue and `D` senators in another.
2. Pop both fronts; the smaller index bans the other and is pushed back as `index + n`.
3. When one queue is empty, the other party wins.

**Cost:** Time O(n) (every comparison bans one senator) · Space O(n) (the two queues)

**Likely follow-ups**
- **Why ban the next opponent?** They're the one who would act soonest against you.
- **Count-based version?** Track pending bans per party while scanning in a loop (circular).

In [12]:
def predict_party_victory(senate):
    """Winning party when each senator bans the next opponent."""
    n = len(senate)
    radiant = deque(i for i, s in enumerate(senate) if s == "R")
    dire = deque(i for i, s in enumerate(senate) if s == "D")
    while radiant and dire:
        r, d = radiant.popleft(), dire.popleft()
        # the earlier senator acts first, bans the other, and returns next round
        if r < d:
            radiant.append(r + n)
        else:
            dire.append(d + n)
    return "Radiant" if radiant else "Dire"


assert predict_party_victory("RD") == "Radiant"
assert predict_party_victory("RDD") == "Dire"
assert predict_party_victory("DDRRR") == "Dire"
print(predict_party_victory("RDD"))

Dire


---
# 2) Intervals

**The gist.** Sort the intervals and overlapping ones end up next to each other — one left-to-right pass decides everything.

> **Think of it like…** meeting cards laid out by start time: overlapping meetings sit side by side. To fit the *most* meetings, always pick the one that ends earliest.

| Goal | Sort by | Why |
|---|---|---|
| merge / insert / union | start | overlapping intervals become neighbours |
| fewest removals, most non-overlapping, arrows | **end** | finishing early leaves the most room |
| max simultaneous overlap (rooms) | starts + a min-heap of ends | count what's active at each moment |

`[a, b]` and `[c, d]` overlap iff `a <= d and c <= b`.

**Spot it:** "merge", "insert", "overlap", "intersection", "minimum rooms / platforms", "remove the fewest", "fewest arrows".

**Classic bugs:** sorting by start when you need earliest end · not deciding whether touching endpoints (`[1,2]`, `[2,3]`) overlap · forgetting `max(end, …)` when merging (a later interval can sit fully inside).

## 2.1 Merging and scheduling

Pick the sort key first (start for merging, end for scheduling); then one pass with "the last kept interval" or a heap of active intervals answers it.

### 2.1.1 [56. Merge Intervals](https://leetcode.com/problems/merge-intervals/) — ★★★ · Medium

**Problem.** Merge all overlapping intervals.  
**Example.** `[[1,3],[2,6],[8,10],[15,18]]` → `[[1,6],[8,10],[15,18]]`  
**Spot it.** Sort by start; overlapping intervals are then consecutive.

**Ask first**
- Do touching intervals (`[1,2]`, `[2,3]`) merge? → Yes here — so use `<=`.

> 💡 **Idea.** Sort by start. Walk through: if an interval starts before the last merged one ends, extend that end; otherwise start a new merged interval.

**Brute force:** repeatedly merge any overlapping pair until none left, O(n²) or worse.

**Steps**
1. Sort by start.
2. If the interval starts within the last merged block, extend its end with `max`.
3. Otherwise start a new block.

**Cost:** Time O(n log n) (sorting) · Space O(n) for the output

**Likely follow-ups**
- **Intervals arrive one at a time (LC 352)?** Keep a sorted structure of disjoint intervals; on insert, find neighbours with bisect and merge. O(log n) + merges.
- **Can't sort / huge coordinate range?** Sweep events (+1 at start, −1 at end); merged blocks are where the count is > 0.
- **Total length covered?** Merge, then sum the lengths.
- **Return the free gaps (employee free time, LC 759)?** Merge everything, then output the gaps between merged intervals.

In [13]:
def merge_intervals(intervals):
    """Union of overlapping intervals."""
    out = []
    for start, end in sorted(intervals):
        # overlaps the last block: extend it (it may be fully inside)
        if out and start <= out[-1][1]:
            out[-1][1] = max(out[-1][1], end)
        else:
            out.append([start, end])
    return out


assert merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]) == [
    [1, 6],
    [8, 10],
    [15, 18],
]
assert merge_intervals([[1, 4], [4, 5]]) == [[1, 5]]
print(merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]))

[[1, 6], [8, 10], [15, 18]]


### 2.1.2 [57. Insert Interval](https://leetcode.com/problems/insert-interval/) — ★★★ · Medium

**Problem.** Insert a new interval into sorted, non-overlapping intervals and merge.  
**Example.** `intervals = [[1,3],[6,9]], new = [2,5]` → `[[1,5],[6,9]]`  
**Spot it.** Three phases: before, overlapping, after.

**Ask first**
- Is the input sorted and non-overlapping? → Yes — that's what allows one pass.

> 💡 **Idea.** Three phases on the sorted list: copy everything that ends before the new one, merge everything that overlaps it, copy the rest.

**Brute force:** append, sort, merge (LC 56) — O(n log n).

**Steps**
1. Copy intervals that end before the new one starts.
2. While intervals start before the new one ends, widen the new interval.
3. Append the widened interval, then the remaining intervals.

**Cost:** Time O(n) · Space O(n)

**Likely follow-ups**
- **Faster than O(n)?** Binary-search where the overlap starts and ends — but building the output is still O(n).
- **Many inserts?** Keep a balanced tree / sorted list of intervals (LC 352 / 715 style).
- **Remove an interval instead (LC 1272)?** Keep the left and right leftover pieces of each overlapping interval.

In [14]:
def insert_interval(intervals, new):
    """Intervals with new inserted and merged."""
    out, i, n = [], 0, len(intervals)
    start, end = new
    # entirely before the new interval
    while i < n and intervals[i][1] < start:
        out.append(intervals[i])
        i += 1
    # overlapping: absorb into the new interval
    while i < n and intervals[i][0] <= end:
        start = min(start, intervals[i][0])
        end = max(end, intervals[i][1])
        i += 1
    out.append([start, end])
    # entirely after
    return out + intervals[i:]


assert insert_interval([[1, 3], [6, 9]], [2, 5]) == [[1, 5], [6, 9]]
assert insert_interval([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]) == [
    [1, 2],
    [3, 10],
    [12, 16],
]
assert insert_interval([], [5, 7]) == [[5, 7]]
print(insert_interval([[1, 3], [6, 9]], [2, 5]))

[[1, 5], [6, 9]]


### 2.1.3 [435. Non-overlapping Intervals](https://leetcode.com/problems/non-overlapping-intervals/) — ★★★ · Medium

**Problem.** Fewest intervals to remove so the rest do not overlap.  
**Example.** `[[1,2],[2,3],[3,4],[1,3]]` → `1` (remove `[1,3]`)  
**Spot it.** Keep the maximum number of non-overlapping intervals → sort by end (activity selection).

**Ask first**
- Do touching intervals overlap? → No, `[1,2]` and `[2,3]` can both stay.

> 💡 **Idea.** Sort by end. Keep every interval that starts after the last kept one ends — finishing earliest leaves the most room. Answer = total − kept.

**Brute force:** try every subset, O(2ⁿ); or DP, O(n²).

**Steps**
1. Sort by end.
2. Keep an interval if it starts at or after the last kept end.
3. Removals = `n − kept`.

**Cost:** Time O(n log n) · Space O(1) besides sorting

**Likely follow-ups**
- **Why sort by end, not start?** A long interval that starts early would block many others.
- **Each interval has a weight (keep the max total weight)?** Weighted interval scheduling: sort by end, DP + binary search for the last compatible interval (LC 1235).
- **Return which intervals to remove?** Everything not chosen by the greedy.

In [15]:
def erase_overlap_intervals(intervals):
    """Minimum removals to eliminate overlaps."""
    kept, last_end = 0, float("-inf")
    # earliest finishing first
    for start, end in sorted(intervals, key=lambda iv: iv[1]):
        # compatible with everything kept so far (touching is fine)
        if start >= last_end:
            kept += 1
            last_end = end
    return len(intervals) - kept


assert erase_overlap_intervals([[1, 2], [2, 3], [3, 4], [1, 3]]) == 1
assert erase_overlap_intervals([[1, 2], [1, 2], [1, 2]]) == 2
assert erase_overlap_intervals([[1, 2], [2, 3]]) == 0
print(erase_overlap_intervals([[1, 2], [2, 3], [3, 4], [1, 3]]))

1


### 2.1.4 [452. Minimum Number of Arrows to Burst Balloons](https://leetcode.com/problems/minimum-number-of-arrows-to-burst-balloons/) — ★★ · Medium

**Problem.** Balloons are intervals on the x-axis; an arrow at `x` bursts all containing `x`. Fewest arrows?  
**Example.** `[[10,16],[2,8],[1,6],[7,12]]` → `2`  
**Spot it.** Same as activity selection: shoot at the earliest end.

**Ask first**
- Does hitting an endpoint burst it? → Yes, inclusive — so `start <= arrow` counts.

> 💡 **Idea.** Sort by end. Shoot an arrow at the first balloon's end; it pops every balloon starting at or before that point. The next balloon starting after it needs a new arrow.

**Brute force:** try arrow positions at every endpoint, O(n²).

**Steps**
1. Sort by end.
2. If a balloon starts after the current arrow, shoot a new arrow at its end.
3. Count arrows.

**Cost:** Time O(n log n) · Space O(1) besides sorting

**Likely follow-ups**
- **Return the arrow positions?** Collect the end of each balloon where you fire.
- **Why does this equal the non-overlapping count?** Each arrow covers one maximal group that all overlap a single point — same greedy as LC 435.
- **Overflow when comparing?** Compare coordinates directly; don't sort with `a − b` comparators in fixed-width languages.

In [16]:
def find_min_arrow_shots(points):
    """Fewest arrows to burst all balloons."""
    arrows, arrow_x = 0, float("-inf")
    for start, end in sorted(points, key=lambda p: p[1]):
        # missed by the last arrow: shoot at this balloon's end
        if start > arrow_x:
            arrows += 1
            arrow_x = end
    return arrows


assert find_min_arrow_shots([[10, 16], [2, 8], [1, 6], [7, 12]]) == 2
assert find_min_arrow_shots([[1, 2], [3, 4], [5, 6], [7, 8]]) == 4
assert find_min_arrow_shots([[1, 2], [2, 3], [3, 4], [4, 5]]) == 2
print(find_min_arrow_shots([[10, 16], [2, 8], [1, 6], [7, 12]]))

2


### 2.1.5 [252. Meeting Rooms](https://leetcode.com/problems/meeting-rooms/) — ★★★ · Easy · Premium

**Problem.** Can one person attend all meetings?  
**Example.** `[[0,30],[5,10],[15,20]]` → `False`; `[[7,10],[2,4]]` → `True`  
**Spot it.** Sort by start and check adjacent overlaps.

**Ask first**
- Can one meeting start exactly when another ends? → Yes, that's fine (`<` not `<=`).

> 💡 **Idea.** Sort by start; if any meeting starts before the previous one ends, someone can't attend all.

**Brute force:** compare every pair, O(n²).

**Steps**
1. Sort by start.
2. Check every meeting ends by the time the next starts.

**Cost:** Time O(n log n) · Space O(n) for the sorted copy

In [17]:
def can_attend_meetings(intervals):
    """True if no meetings overlap."""
    intervals = sorted(intervals)
    # each meeting must end before the next one starts
    return all(a[1] <= b[0] for a, b in zip(intervals, intervals[1:]))


assert not can_attend_meetings([[0, 30], [5, 10], [15, 20]])
assert can_attend_meetings([[7, 10], [2, 4]])
print(can_attend_meetings([[7, 10], [2, 4]]))

True


### 2.1.6 [253. Meeting Rooms II](https://leetcode.com/problems/meeting-rooms-ii/) — ★★★ · Medium · Premium

**Problem.** Minimum number of meeting rooms needed.  
**Example.** `[[0,30],[5,10],[15,20]]` → `2`  
**Spot it.** Max number of simultaneously active meetings → min-heap of end times.

**Ask first**
- Does a meeting ending at t free its room for one starting at t? → Yes — pop when `end <= start`.

> 💡 **Idea.** Sort by start. Keep a min-heap of end times for rooms in use. For each meeting, free the room that ends earliest if it's done; push this meeting's end. Heap size at the peak = rooms needed.

**Brute force:** for every moment, count overlapping meetings, O(n²).

**Steps**
1. Sort by start.
2. If the earliest ending room is free by this start, reuse it (replace its end).
3. Otherwise push a new end (a new room).
4. The heap size is the number of rooms.

**Cost:** Time O(n log n) · Space O(n)

**Likely follow-ups**
- **Without a heap?** Sort starts and ends separately; two pointers counting +1/−1 (sweep line).
- **Which room does each meeting get (LC 2402)?** Two heaps: free rooms by id, busy rooms by `(end, id)`.
- **The busiest moment / max overlap point?** Sweep line, record where the count peaks — or for 2D rectangles, a 2D difference array (LC 2536).
- **Can a meeting be delayed if no room is free?** Same two-heap simulation, pushing start time to the earliest free end.

In [18]:
def min_meeting_rooms(intervals):
    """Rooms needed for all meetings."""
    # end times of rooms in use; earliest free room on top
    ends = []
    for start, end in sorted(intervals):
        # earliest room is free: reuse it
        if ends and ends[0] <= start:
            heapq.heapreplace(ends, end)
        # every room is busy: open another
        else:
            heapq.heappush(ends, end)
    return len(ends)


assert min_meeting_rooms([[0, 30], [5, 10], [15, 20]]) == 2
assert min_meeting_rooms([[7, 10], [2, 4]]) == 1
print(min_meeting_rooms([[0, 30], [5, 10], [15, 20]]))

2


### 2.1.7 [986. Interval List Intersections](https://leetcode.com/problems/interval-list-intersections/) — ★ · Medium

**Problem.** Intersections of two sorted, disjoint interval lists.  
**Example.** `A = [[0,2],[5,10]], B = [[1,5],[8,12]]` → `[[1,2],[5,5],[8,10]]`  
**Spot it.** Two pointers; advance the interval that ends first.

**Ask first**
- Closed intervals? → Yes, a single shared point counts.

> 💡 **Idea.** Two pointers, one per list. The overlap of the current pair is `[max(starts), min(ends)]` if non-empty. Then move the pointer whose interval ends first.

**Brute force:** compare every pair, O(m·n).

**Steps**
1. Overlap of the current pair = `[max starts, min ends]` if non-empty.
2. Record it.
3. Advance the pointer whose interval ends first.

**Cost:** Time O(m + n) · Space O(m + n) for the output

**Likely follow-ups**
- **Union instead of intersection?** Merge both sorted lists then merge overlaps (LC 56).
- **k lists?** Intersect pairwise, or sweep all events counting how many lists are active (= k).
- **Lists unsorted?** Sort first, O(n log n).

In [19]:
def interval_intersection(a, b):
    """Pairwise intersections of two interval lists."""
    out, i, j = [], 0, 0
    while i < len(a) and j < len(b):
        # overlap of the current pair (if any)
        lo = max(a[i][0], b[j][0])
        hi = min(a[i][1], b[j][1])
        if lo <= hi:
            out.append([lo, hi])
        # the one ending first is finished
        if a[i][1] < b[j][1]:
            i += 1
        else:
            j += 1
    return out


a = [[0, 2], [5, 10], [13, 23], [24, 25]]
b = [[1, 5], [8, 12], [15, 24], [25, 26]]
assert interval_intersection(a, b) == [
    [1, 2],
    [5, 5],
    [8, 10],
    [15, 23],
    [24, 24],
    [25, 25],
]
print(interval_intersection(a, b))

[[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]


### 2.1.8 [729. My Calendar I](https://leetcode.com/problems/my-calendar-i/) — ★ · Medium

**Problem.** `book(start, end)` succeeds if it does not overlap any existing booking.  
**Example.** `book(10, 20)` → `True`, `book(15, 25)` → `False`, `book(20, 30)` → `True`  
**Spot it.** Keep bookings sorted; only the neighbours of the insertion point can overlap.

**Ask first**
- Half-open `[start, end)`? → Yes, so touching bookings are fine.

> 💡 **Idea.** Keep bookings in a sorted list. For a new `[s, e)`, binary-search its spot and check it doesn't overlap the neighbour before or after.

**Brute force:** check against every booking, O(n) per call.

**Steps**
1. Find the insertion index with `bisect_left`.
2. The previous booking must end by `start`.
3. The next booking must start at or after `end`.
4. If both hold, insert and return `True`.

**Cost:** Time O(log n) search, O(n) insert (O(log n) with a balanced tree) · Space O(n)

**Likely follow-ups**
- **Allow double bookings but not triple (LC 731)?** Keep a list of single bookings and a list of overlaps; reject if the new one hits an overlap.
- **Max overlap count after each booking (LC 732)?** Sweep line over a sorted map of +1/−1 deltas (or a segment tree).
- **O(log n) insert?** Balanced BST / `SortedList` keyed by start.

In [20]:
class MyCalendar:
    """Non-overlapping booking calendar."""

    def __init__(self):
        # sorted, non-overlapping (start, end) pairs
        self.bookings = []

    def book(self, start, end):
        i = bisect_left(self.bookings, (start, end))
        # clash with the booking before
        if i > 0 and self.bookings[i - 1][1] > start:
            return False
        # clash with the booking after
        if i < len(self.bookings) and self.bookings[i][0] < end:
            return False
        insort(self.bookings, (start, end))
        return True


cal = MyCalendar()
assert [cal.book(10, 20), cal.book(15, 25), cal.book(20, 30)] == [True, False, True]
print(cal.bookings)

[(10, 20), (20, 30)]


---
# 3) Math and Geometry

**The gist.** These hide a formula or a pattern. Simulating is too slow or overflows; the trick is to spot the structure (digits, factors, powers of two, corners) and compute it directly.

> **Think of it like…** counting the zeros at the end of 100!: multiplying it out is hopeless, but each zero needs a factor 5 — three divisions and you're done.

**Spot it:** digits of a number (reverse, palindrome, plus one) · huge n → O(log n) (fast power, halving) · "uniformly random" → Fisher–Yates.

**Classic bugs:** 32-bit overflow rules (Python never overflows — clamp yourself) · Python's `-7 // 2 == -4` and `-7 % 10 == 3` · biased shuffle (picking `j` from the whole array).

## 3.1 Number manipulation

Tools: `divmod(x, 10)` to peel digits, carries from the right, `gcd`, repeated squaring / halving for O(log n).

### 3.1.1 [7. Reverse Integer](https://leetcode.com/problems/reverse-integer/) — ★★ · Medium

**Problem.** Reverse the digits of a 32-bit signed integer; return 0 on overflow.  
**Example.** `123` → `321`; `-123` → `-321`  
**Spot it.** Pop digits with `% 10` and `// 10`, check the 32-bit range.

**Ask first**
- Can I use 64-bit ints? → No — check overflow *before* it happens.
- Negatives? → Keep the sign, reverse the absolute value.

> 💡 **Idea.** Peel digits off the end and push them onto the result, then check the 32-bit range. (Python can check at the end; in fixed-width languages check *before* each push — see follow-ups.)

**Brute force:** convert to a string, reverse, convert back, then check range — works in Python, not the point of the question.

**Steps**
1. Remember the sign and work with `abs(x)`.
2. Peel the last digit and append it to `rev`.
3. Restore the sign.
4. Out of 32-bit range → 0.

**Cost:** Time O(digits) · Space O(1)

**Likely follow-ups**
- **Check overflow without a bigger type?** Before `res = res * 10 + d`: if `res > MAX // 10` or (`res == MAX // 10` and `d > 7`), return 0 (mirror for negatives with 8).
- **Palindrome number (LC 9) without a string?** Reverse only half the digits and compare with the other half.

In [21]:
def reverse_integer(x):
    """Digit-reversed x, or 0 on 32-bit overflow."""
    sign = -1 if x < 0 else 1
    x, rev = abs(x), 0
    while x:
        # take the last digit off x and append it to rev
        x, digit = divmod(x, 10)
        rev = rev * 10 + digit
    rev *= sign
    return rev if -(2**31) <= rev <= 2**31 - 1 else 0


assert reverse_integer(123) == 321
assert reverse_integer(-123) == -321
assert reverse_integer(120) == 21
assert reverse_integer(1534236469) == 0
print(reverse_integer(-123))

-321


### 3.1.2 [66. Plus One](https://leetcode.com/problems/plus-one/) — ★★ · Easy

**Problem.** Digits array of a number; add one.  
**Example.** `[1, 2, 9]` → `[1, 3, 0]`  
**Spot it.** Carry from the end.

**Ask first**
- All 9s? → The result is one digit longer (`[9,9]` → `[1,0,0]`).

> 💡 **Idea.** Add 1 at the end; while a digit becomes 10, set it to 0 and carry left. If the carry falls off the front, prepend a 1.

**Brute force:** convert to an int, add 1, convert back — fine in Python, not in fixed-width languages.

**Steps**
1. Walk from the last digit.
2. A digit < 9: increment and return.
3. A 9: set it to 0 and keep carrying.
4. All 9s: prepend 1.

**Cost:** Time O(n) · Space O(n) for the copy

In [22]:
def plus_one(digits):
    """Digits of the number plus one."""
    digits = digits.copy()
    for i in range(len(digits) - 1, -1, -1):
        # no carry needed beyond this digit
        if digits[i] < 9:
            digits[i] += 1
            return digits
        # 9 + 1 = 10: write 0, carry on
        digits[i] = 0
    # every digit was 9
    return [1] + digits


assert plus_one([1, 2, 3]) == [1, 2, 4]
assert plus_one([9, 9]) == [1, 0, 0]
print(plus_one([4, 3, 2, 1]))

[4, 3, 2, 2]


### 3.1.3 [202. Happy Number](https://leetcode.com/problems/happy-number/) — ★★ · Easy

**Problem.** Repeatedly replace `n` by the sum of squares of its digits; does it reach 1?  
**Example.** `19` → `True` (19 → 82 → 68 → 100 → 1)  
**Spot it.** The sequence either hits 1 or cycles → cycle detection.

**Ask first**
- Does a loop that never hits 1 mean False? → Yes.

> 💡 **Idea.** Keep replacing n with the sum of squared digits. You either reach 1 or loop forever — detect the loop with a set (or slow/fast pointers).

**Brute force:** run a fixed big number of steps and hope — unreliable.

**Steps**
1. `nxt(x)` = sum of squared digits.
2. Slow moves one step, fast two steps.
3. Stop when fast is 1 (happy) or slow meets fast (cycle).

**Cost:** Time O(log n) per step, few steps · Space O(1)

In [23]:
def is_happy(n):
    """True if n is a happy number."""

    def nxt(x):
        """Sum of squared digits."""
        return sum(int(d) ** 2 for d in str(x))

    # tortoise and hare on the sequence n, nxt(n), ...
    slow, fast = n, nxt(n)
    while fast != 1 and slow != fast:
        slow, fast = nxt(slow), nxt(nxt(fast))
    return fast == 1


assert is_happy(19)
assert not is_happy(2)
print(is_happy(19))

True


### 3.1.4 [50. Pow(x, n)](https://leetcode.com/problems/powx-n/) — ★★ · Medium

**Problem.** Compute `xⁿ` for integer `n` (possibly negative).  
**Example.** `x = 2.0, n = 10` → `1024.0`  
**Spot it.** Fast exponentiation by squaring.

**Ask first**
- Negative n? → Use `1/x` and `−n`.
- n = −2³¹? → Negating it overflows in fixed-width languages; use a 64-bit n or handle it separately.

> 💡 **Idea.** Fast power: `xⁿ = (x²)^(n/2)`, times one extra x if n is odd. Halve n each step → O(log n). For negative n, compute `(1/x)^(-n)`.

**Brute force:** multiply x by itself n times, O(n).

**Steps**
1. Negative `n` → use `1/x` and `−n`.
2. While `n > 0`: if the lowest bit is set, multiply the result by `x`.
3. Square `x`, shift `n` right.

**Cost:** Time O(log n) · Space O(1)

**Likely follow-ups**
- **Iterative version?** Loop over the bits of n: if the bit is set, multiply the result by the current power; square the power each step.
- **`xⁿ mod m` for huge n?** Same loop with `% m` after each multiply (modular exponentiation).
- **Matrix power (Fibonacci in O(log n))?** Same squaring idea on 2×2 matrices.

In [24]:
def my_pow(x, n):
    """x raised to n."""
    if n < 0:
        x, n = 1 / x, -n
    result = 1.0
    while n:
        # this bit of n contributes the current power of x
        if n & 1:
            result *= x
        # x, x^2, x^4, x^8, ...
        x *= x
        n >>= 1
    return result


assert abs(my_pow(2.0, 10) - 1024.0) < 1e-9
assert abs(my_pow(2.1, 3) - 9.261) < 1e-9
assert abs(my_pow(2.0, -2) - 0.25) < 1e-9
print(my_pow(2.0, 10))

1024.0


### 3.1.5 [43. Multiply Strings](https://leetcode.com/problems/multiply-strings/) — ★★ · Medium

**Problem.** Multiply two non-negative numbers given as strings, without big-int conversion.  
**Example.** `"12"` × `"12"` → `"144"`  
**Spot it.** Grade-school multiplication: digit `i` × digit `j` lands in positions `i + j` and `i + j + 1`.

**Ask first**
- Can I convert to int? → No.
- Either number `"0"`? → Return `"0"`.

> 💡 **Idea.** Grade-school multiplication: digit `i` of one number times digit `j` of the other lands at positions `i + j` and `i + j + 1` of the result. Add, carry, strip leading zeros.

**Brute force:** repeated addition, O(value) — way too slow.

**Steps**
1. Result array of `m + n` zeros.
2. For each digit pair from the right, add the product to `res[i + j + 1]`.
3. Keep the ones digit there; add the carry to `res[i + j]`.
4. Join and strip leading zeros (empty → `"0"`).

**Cost:** Time O(mn) · Space O(m + n)

**Likely follow-ups**
- **Add two number strings (LC 415)?** Walk from the right with a carry.
- **Very long numbers (10⁵ digits)?** Karatsuba (O(n^1.58)) or FFT (O(n log n)) — mention only.
- **Negative numbers?** Handle the sign separately, multiply magnitudes.

In [25]:
def multiply(num1, num2):
    """Product of two decimal strings."""
    # an m-digit times an n-digit number has at most m + n digits
    res = [0] * (len(num1) + len(num2))
    for i in range(len(num1) - 1, -1, -1):
        for j in range(len(num2) - 1, -1, -1):
            # this digit product lands at i + j + 1, carrying into i + j
            total = int(num1[i]) * int(num2[j]) + res[i + j + 1]
            res[i + j + 1] = total % 10
            res[i + j] += total // 10
    out = "".join(map(str, res)).lstrip("0")
    return out or "0"


assert multiply("2", "3") == "6"
assert multiply("123", "456") == "56088"
assert multiply("0", "52") == "0"
print(multiply("123", "456"))

56088


## 3.2 Geometry and randomness

Geometry: fix one point (a corner) so the rest is counting. Randomness: Fisher–Yates for a fair shuffle.

### 3.2.1 [2013. Detect Squares](https://leetcode.com/problems/detect-squares/) — ★★ · Medium

**Problem.** `add(point)`; `count(point)` = number of axis-aligned squares with the query point as one corner and three added points as the others (duplicates count separately).  
**Example.** `add([3,10])`, `add([11,2])`, `add([3,2])`, `count([11,10])` → `1`, `count([14,8])` → `0`  
**Spot it.** Choose the diagonal corner; the other two corners are then determined.

**Ask first**
- Duplicate points count separately? → Yes — multiply by their counts.
- Only axis-aligned squares with positive area? → Yes.

> 💡 **Idea.** Store point counts. For a query point, any stored point on its *diagonal* (`|dx| == |dy| > 0`) is the opposite corner of a square; multiply the counts of that corner and the two remaining corners.

**Brute force:** check every triple of stored points, O(n³) per query.

**Steps**
1. Store point counts.
2. For each stored `(x, y)` with `|x − qx| == |y − qy| > 0` (a diagonal):
3. add `count(x, y) · count(qx, y) · count(x, qy)`.

**Cost:** Time O(1) add, O(distinct points) count · Space O(distinct points)

**Likely follow-ups**
- **Faster per query?** Index points by x-column: only scan points sharing the query's x (O(points in that column)).
- **Rectangles, not squares?** For each stored diagonal point, check the other two corners exist (LC 939 style).
- **Points can be removed?** Decrement counts; drop keys at 0.

In [26]:
class DetectSquares:
    """Count axis-aligned squares through a query point."""

    def __init__(self):
        self.points = Counter()

    def add(self, point):
        self.points[tuple(point)] += 1

    def count(self, point):
        qx, qy = point
        total = 0
        for (x, y), n in self.points.items():
            # (x, y) is diagonal to the query: a non-degenerate square
            if abs(x - qx) == abs(y - qy) and x != qx:
                # the two remaining corners are forced
                total += n * self.points[(qx, y)] * self.points[(x, qy)]
        return total


ds = DetectSquares()
for p in [[3, 10], [11, 2], [3, 2]]:
    ds.add(p)
assert ds.count([11, 10]) == 1 and ds.count([14, 8]) == 0
ds.add([11, 2])
assert ds.count([11, 10]) == 2
print("ok")

ok


### 3.2.2 [384. Shuffle an Array](https://leetcode.com/problems/shuffle-an-array/) — ★ · Medium

**Problem.** `shuffle()` returns a uniformly random permutation; `reset()` restores the original.  
**Example.** `[1, 2, 3]`: `shuffle()` → for example `[3, 1, 2]`, then `reset()` → `[1, 2, 3]`  
**Spot it.** Fisher–Yates shuffle.

**Ask first**
- Must every permutation be equally likely? → Yes — so pick `j` from `0..i`, not from the whole array.

> 💡 **Idea.** Fisher–Yates: for i from the end down to 1, swap `a[i]` with `a[random index in 0..i]`. Every permutation is equally likely.

**Brute force:** sort by random keys, O(n log n) — fine but slower.

**Steps**
1. Copy the original.
2. For `i` from n − 1 down to 1, pick `j` uniformly in `0..i`.
3. Swap `a[i]` and `a[j]`.
4. `reset` returns a copy of the original.

**Cost:** Time O(n) per shuffle · Space O(n)

**Likely follow-ups**
- **Why is picking from the whole array biased?** It produces nⁿ equally likely outcomes, which can't split evenly over n! permutations.
- **Pick k random items from a stream (reservoir sampling)?** Keep the first k; for item i, replace a random slot with probability k / i.
- **Shuffle without changing the original?** Copy first (that's what `reset` relies on).

In [27]:
class Shuffler:
    """Uniform random shuffling with reset."""

    def __init__(self, nums):
        self.original = nums.copy()

    def reset(self):
        return self.original.copy()

    def shuffle(self):
        a = self.original.copy()
        for i in range(len(a) - 1, 0, -1):
            # choose a[i] uniformly from the not-yet-placed prefix a[0..i]
            j = random.randint(0, i)
            a[i], a[j] = a[j], a[i]
        return a


random.seed(0)
sh = Shuffler([1, 2, 3])
counts = Counter(tuple(sh.shuffle()) for _ in range(6000))
assert len(counts) == 6 and min(counts.values()) > 850
assert sh.reset() == [1, 2, 3]
print(counts)

Counter({(3, 1, 2): 1027, (2, 3, 1): 1008, (2, 1, 3): 1003, (1, 3, 2): 993, (1, 2, 3): 989, (3, 2, 1): 980})


---
# 4) Bit Manipulation

**The gist.** Treat a number as a row of on/off switches and use AND, OR, XOR and shifts to read or flip them — often replacing a hash map or a loop with one operation.

> **Think of it like…** light switches: XOR flips a switch (twice = back off), AND with a mask looks at only some switches, a shift slides the whole row.

| Trick | Meaning |
|---|---|
| `x & 1` | lowest bit (odd?) |
| `x & (x − 1)` | clear the lowest set bit (power of two ⇔ result is 0) |
| `x & −x` | isolate the lowest set bit |
| `a ^ a = 0`, `a ^ 0 = a` | XOR cancels pairs |
| `x >> k & 1` | k-th bit |

Python ints never overflow, so use a mask `0xFFFFFFFF` when a problem relies on 32-bit wrap-around.

**Spot it:** "every element appears twice except one", "missing number", O(1) space on counting/pairing, "number of 1 bits", "without using + / −".

**Classic bugs:** Python negatives (`>>` never reaches 0; no 32-bit wrap) · precedence: `a & b == 0` means `a & (b == 0)` — use brackets · forgetting `n > 0` in the power-of-two test.

## 4.1 XOR and bit counting

Two families: XOR cancellation (pairs vanish, leaving the unpaired one) and per-bit reasoning (look at one bit position at a time).

### 4.1.1 [136. Single Number](https://leetcode.com/problems/single-number/) — ★★ · Easy

**Problem.** Every element appears twice except one; find it in O(1) space.  
**Example.** `[4, 1, 2, 1, 2]` → `4`  
**Spot it.** XOR cancels pairs.

**Ask first**
- O(1) extra space? → Yes, that's why XOR.

> 💡 **Idea.** XOR everything — pairs cancel to 0, the single one is left.

**Brute force:** count with a dict, O(n) space.

**Steps**
1. Start with 0.
2. XOR every number into it.
3. Return the result.

**Cost:** Time O(n) · Space O(1)

In [28]:
def single_number(nums):
    """The element without a pair."""
    out = 0
    for x in nums:
        # pairs cancel: a ^ a = 0
        out ^= x
    return out


assert single_number([2, 2, 1]) == 1
assert single_number([4, 1, 2, 1, 2]) == 4
print(single_number([4, 1, 2, 1, 2]))

4


### 4.1.2 [268. Missing Number](https://leetcode.com/problems/missing-number/) — ★★★ · Easy

**Problem.** `n` distinct numbers from `0..n`; find the missing one.  
**Example.** `[3, 0, 1]` → `2`  
**Spot it.** XOR indices and values (or use the sum formula).

**Ask first**
- Values in `0..n`, all distinct? → Yes.

> 💡 **Idea.** XOR all indices `0..n` and all values; everything paired cancels, leaving the missing number. (Or: expected sum − actual sum.)

**Brute force:** sort and scan, O(n log n); or a set, O(n) space.

**Steps**
1. Start with `n` (the index that has no slot).
2. XOR in every index `i` and value `x`.
3. Return the result.

**Cost:** Time O(n) · Space O(1)

In [29]:
def missing_number(nums):
    """The value in 0..n not present."""
    # n has no index of its own, so seed with it
    out = len(nums)
    for i, x in enumerate(nums):
        # every present value cancels its matching index
        out ^= i ^ x
    return out


assert missing_number([3, 0, 1]) == 2
assert missing_number([0, 1]) == 2
assert missing_number([9, 6, 4, 2, 3, 5, 7, 0, 1]) == 8
print(missing_number([3, 0, 1]))

2


### 4.1.3 [191. Number of 1 Bits](https://leetcode.com/problems/number-of-1-bits/) — ★★★ · Easy

**Problem.** Count set bits (Hamming weight).  
**Example.** `11` (binary `1011`) → `3`  
**Spot it.** `n & (n − 1)` drops the lowest set bit.

**Ask first**
- Called many times? → Precompute a lookup table for each byte.

> 💡 **Idea.** `n & (n − 1)` clears the lowest 1 bit — count how many times until n is 0.

**Brute force:** check all 32 bits one by one.

**Steps**
1. While `n` is non-zero, clear its lowest set bit.
2. Count the iterations.

**Cost:** Time O(set bits) · Space O(1)

In [30]:
def hamming_weight(n):
    """Number of 1 bits."""
    count = 0
    while n:
        # removes exactly the lowest set bit
        n &= n - 1
        count += 1
    return count


assert hamming_weight(11) == 3
assert hamming_weight(128) == 1
assert hamming_weight(2147483645) == 30
print(hamming_weight(11))

3


### 4.1.4 [338. Counting Bits](https://leetcode.com/problems/counting-bits/) — ★★★ · Easy

**Problem.** For every `i` in `0..n`, the number of set bits.  
**Example.** `n = 5` → `[0, 1, 1, 2, 1, 2]`  
**Spot it.** Reuse smaller answers: `bits[i] = bits[i >> 1] + (i & 1)`.

**Ask first**
- O(n) without counting each number from scratch? → Yes, reuse earlier answers.

> 💡 **Idea.** `bits[i] = bits[i >> 1] + (i & 1)` — i has the same bits as i // 2, plus its last bit.

**Brute force:** count bits of every number, O(n log n).

**Steps**
1. `bits[0] = 0`.
2. For each `i`, `bits[i] = bits[i >> 1] + (i & 1)`.

**Cost:** Time O(n) · Space O(n) for the output

In [31]:
def count_bits(n):
    """Set-bit counts for 0..n."""
    bits = [0] * (n + 1)
    for i in range(1, n + 1):
        # i without its last bit, plus that last bit
        bits[i] = bits[i >> 1] + (i & 1)
    return bits


assert count_bits(2) == [0, 1, 1]
assert count_bits(5) == [0, 1, 1, 2, 1, 2]
print(count_bits(5))

[0, 1, 1, 2, 1, 2]


### 4.1.5 [190. Reverse Bits](https://leetcode.com/problems/reverse-bits/) — ★★★ · Easy

**Problem.** Reverse the bits of a 32-bit unsigned integer.  
**Example.** `43261596` → `964176192` (the 32 bits are flipped end to end)  
**Spot it.** Shift bits out of `n` and into the result, 32 times.

**Ask first**
- Unsigned 32-bit? → Yes, so always do exactly 32 steps.

> 💡 **Idea.** 32 times: shift the result left, add the lowest bit of n, shift n right.

**Brute force:** convert to a 32-char binary string, reverse, parse.

**Steps**
1. Repeat 32 times:
2. `out = (out << 1) | (n & 1)`.
3. `n >>= 1`.

**Cost:** Time O(32) · Space O(1)

In [32]:
def reverse_bits(n):
    """32-bit reversal of n."""
    out = 0
    for _ in range(32):
        # take n's lowest bit, push it onto out
        out = (out << 1) | (n & 1)
        n >>= 1
    return out


assert reverse_bits(0b00000010100101000001111010011100) == 964176192
assert reverse_bits(0b11111111111111111111111111111101) == 3221225471
print(reverse_bits(43261596))

964176192


### 4.1.6 [371. Sum of Two Integers](https://leetcode.com/problems/sum-of-two-integers/) — ★★★ · Medium

**Problem.** Add two integers without `+` or `−`.  
**Example.** `a = 1, b = 2` → `3`  
**Spot it.** XOR adds without carry; AND shifted left is the carry.

**Ask first**
- Negative numbers? → Yes — Python ints are unbounded, so mask with `0xFFFFFFFF` and fix the sign at the end.

> 💡 **Idea.** `a ^ b` adds without carries; `(a & b) << 1` is the carry. Repeat until the carry is 0. In Python, mask to 32 bits and convert back for negatives.

**Brute force:** use `+` — not allowed.

**Steps**
1. While there is a carry: `a, b = a ^ b, (a & b) << 1`, both masked to 32 bits.
2. Mask the result.
3. Convert from two's complement if bit 31 is set.

**Cost:** Time O(32) · Space O(1)

**Likely follow-ups**
- **Subtract without `−`?** `a + (~b + 1)` (add the two's complement).
- **Why does the loop end?** The carry moves left each round and falls off after at most 32 rounds (with the mask).
- **Multiply without `*`?** Shift-and-add over the bits of one number.

In [33]:
def get_sum(a, b):
    """a + b using only bit operations."""
    mask = 0xFFFFFFFF
    while b & mask:
        # sum without carries, and the carries shifted into place
        a, b = (a ^ b) & mask, ((a & b) << 1) & mask
    a &= mask
    # reinterpret as signed 32-bit
    return a if a < 1 << 31 else a - (1 << 32)


assert get_sum(1, 2) == 3
assert get_sum(2, 3) == 5
assert get_sum(-1, 1) == 0
assert get_sum(-12, -8) == -20
print(get_sum(-12, -8))

-20


### 4.1.7 [201. Bitwise AND of Numbers Range](https://leetcode.com/problems/bitwise-and-of-numbers-range/) — ★★ · Medium

**Problem.** Bitwise AND of all integers in `[left, right]`.  
**Example.** `left = 5, right = 7` → `4`  
**Spot it.** The result is the common binary prefix of `left` and `right`.

**Ask first**
- Inclusive range? → Yes.

> 💡 **Idea.** The AND of a range keeps only the common binary prefix of `left` and `right`. Shift both right until equal, then shift back.

**Brute force:** AND every number in the range, O(right − left).

**Steps**
1. Shift both right until they are equal, counting shifts.
2. Shift the common prefix back left.

**Cost:** Time O(32) · Space O(1)

**Likely follow-ups**
- **Another way?** `while right > left: right &= right − 1` — keeps clearing the lowest bit of right.
- **Bitwise OR of a range?** Common prefix, then all lower bits set to 1.

In [34]:
def range_bitwise_and(left, right):
    """AND of all numbers in [left, right]."""
    shift = 0
    # drop differing low bits until only the common prefix is left
    while left != right:
        left >>= 1
        right >>= 1
        shift += 1
    return left << shift


assert range_bitwise_and(5, 7) == 4
assert range_bitwise_and(0, 0) == 0
assert range_bitwise_and(1, 2147483647) == 0
print(range_bitwise_and(5, 7))

4


---
# 5) Design

**The gist.** Build a class where each operation hits a target cost. List the operations and their costs first, then pick one structure per need and make them point at each other.

> **Think of it like…** a library: a catalogue (hash map) to find a book instantly, shelves (array) to grab a random one, a returns cart (queue) for arrival order. Each structure does one job.

| Need | Structure |
|---|---|
| O(1) lookup by key | hash map |
| O(1) random pick | array (+ map value → index) |
| O(1) at both ends, sliding windows | deque / ring buffer |
| O(1) move/remove given a node | doubly linked list |
| O(log n) min / max | heap |
| "largest ≤ t" queries | sorted list + bisect |

**Spot it:** "design a class", "O(1) average for each operation", "iterator", "in the last 5 minutes".

**Classic bugs:** an operation that quietly costs O(n) (removing from the middle of a list, scanning a window) · forgetting to update *both* structures when they point at each other.

## 5.1 Data-structure design

Before coding, write `operation → required cost → structure`. Most bugs come from one operation that silently costs O(n).

### 5.1.1 [380. Insert Delete GetRandom O(1)](https://leetcode.com/problems/insert-delete-getrandom-o1/) — ★★ · Medium

**Problem.** `insert`, `remove` and `get_random` (uniform), all average O(1).  
**Example.** `insert(1)` → `True`, `insert(2)` → `True`, `insert(1)` → `False`, `remove(1)` → `True`, `get_random()` → `2`  
**Spot it.** Array for random access + hash map value → index; delete by swapping with the last element.

**Ask first**
- Duplicates allowed? → Not here. With duplicates (LC 381), map each value to a *set* of indices.

> 💡 **Idea.** Array for O(1) random picks + map value → index. To remove, swap the value with the last element, fix its index in the map, and pop the end.

**Brute force:** a set (no O(1) random pick) or a list (O(n) remove).

**Steps**
1. `values` array and `index` map value → position.
2. `insert`: append and record the index.
3. `remove`: copy the last value into the removed slot, update its index, pop the end, delete the key.
4. `get_random`: `random.choice(values)`.

**Cost:** Time O(1) average per operation · Space O(n)

**Likely follow-ups**
- **Allow duplicates (LC 381)?** Map value → set of indices; remove any one index, swap with last, update both sets.
- **Weighted random pick?** Prefix sums + binary search (LC 528), or a Fenwick tree if weights change.
- **Thread-safe?** One lock around each operation (they all touch both structures).

In [35]:
class RandomizedSet:
    """Set with O(1) insert, remove and uniform sampling."""

    def __init__(self):
        # compact array for sampling; value -> position for lookups
        self.values = []
        self.index = {}

    def insert(self, val):
        if val in self.index:
            return False
        self.index[val] = len(self.values)
        self.values.append(val)
        return True

    def remove(self, val):
        if val not in self.index:
            return False
        # move the last value into the hole and fix its index
        i, last = self.index[val], self.values[-1]
        self.values[i], self.index[last] = last, i
        # removing from the end is O(1)
        self.values.pop()
        del self.index[val]
        return True

    def get_random(self):
        return random.choice(self.values)


rs = RandomizedSet()
assert rs.insert(1) and not rs.remove(2) and rs.insert(2)
assert rs.get_random() in (1, 2)
assert rs.remove(1) and not rs.insert(2) and rs.get_random() == 2
print(rs.values)

[2]


### 5.1.2 [362. Design Hit Counter](https://leetcode.com/problems/design-hit-counter/) — ★★ · Medium · Premium

**Problem.** `hit(timestamp)` and `get_hits(timestamp)` = hits in the past 300 seconds; timestamps are non-decreasing.  
**Example.** `hit(1)`, `hit(2)`, `hit(3)`, `get_hits(4)` → `3`, `hit(300)`, `get_hits(300)` → `4`, `get_hits(301)` → `3`  
**Spot it.** Queue of timestamps; evict entries older than the window.

**Ask first**
- Do timestamps arrive in order? → Yes (non-decreasing).
- Many hits per second? → Then store `(timestamp, count)` pairs instead of one entry per hit.

> 💡 **Idea.** Keep a queue of hit timestamps; on each call, drop timestamps older than 300 seconds and return the queue size.

**Brute force:** store every hit forever and scan all of them per call.

**Steps**
1. `hit` appends the timestamp.
2. `get_hits` pops from the front while the oldest is ≤ `timestamp − 300`.
3. Return the queue length.

**Cost:** Time Amortised O(1) (each hit is appended and popped once) · Space O(hits in the window)

**Likely follow-ups**
- **Huge number of hits per second?** Fixed 300-slot ring buffer: `times[t % 300]` and `counts[t % 300]`; reset a slot when its time is stale. O(300) per query, O(1) memory.
- **Many machines / distributed?** Each machine keeps its own counter; sum on query (or push per-second counts to a central store).
- **Thread-safe?** Lock per bucket, or atomic counters in the ring buffer.

In [36]:
class HitCounter:
    """Hits within the last 300 seconds."""

    def __init__(self):
        # timestamps in arrival (non-decreasing) order
        self.hits = deque()

    def hit(self, timestamp):
        self.hits.append(timestamp)

    def get_hits(self, timestamp):
        # evict hits that fell out of the 300-second window
        while self.hits and self.hits[0] <= timestamp - 300:
            self.hits.popleft()
        return len(self.hits)


hc = HitCounter()
for t in (1, 2, 3):
    hc.hit(t)
assert hc.get_hits(4) == 3
hc.hit(300)
assert hc.get_hits(300) == 4 and hc.get_hits(301) == 3
print(hc.get_hits(301))

3


### 5.1.3 [622. Design Circular Queue](https://leetcode.com/problems/design-circular-queue/) — ★★ · Medium

**Problem.** Fixed-capacity FIFO queue on an array with `en_queue`, `de_queue`, `front`, `rear`.  
**Example.** capacity 3: `en_queue(1)`, `en_queue(2)`, `en_queue(3)` → all `True`, `en_queue(4)` → `False` (full), `rear()` → `3`, `de_queue()` → `True`  
**Spot it.** Ring buffer: head index + size, positions taken modulo capacity.

**Ask first**
- Fixed capacity? → Yes, `k` slots.

> 💡 **Idea.** A fixed-size array with a `head` index and a `count`. The tail is `(head + count − 1) % k`; enqueue writes at `(head + count) % k`.

**Brute force:** a Python list with `pop(0)` — O(n) dequeue.

**Steps**
1. Store `data`, `head` and `size`.
2. `en_queue`: if not full, write at `(head + size) % k` and grow `size`.
3. `de_queue`: if not empty, advance `head` modulo k and shrink `size`.
4. `front` = `data[head]`, `rear` = `data[(head + size − 1) % k]`.

**Cost:** Time O(1) per operation · Space O(k)

**Likely follow-ups**
- **Why keep `count` instead of `head == tail` checks?** With only two indices, "empty" and "full" look the same.
- **Thread-safe / producer–consumer?** Lock + condition variables (blocking queue), or a lock-free single-producer single-consumer ring.
- **Circular deque (LC 641)?** Same ring with `head` moving both ways.

In [37]:
class MyCircularQueue:
    """Ring-buffer queue."""

    def __init__(self, k):
        self.data = [0] * k
        # size separates "full" from "empty"
        self.head = self.size = 0

    def en_queue(self, value):
        if self.size == len(self.data):
            return False
        # the slot just after the last element, wrapping around
        self.data[(self.head + self.size) % len(self.data)] = value
        self.size += 1
        return True

    def de_queue(self):
        if not self.size:
            return False
        self.head = (self.head + 1) % len(self.data)
        self.size -= 1
        return True

    def front(self):
        return self.data[self.head] if self.size else -1

    def rear(self):
        return (
            self.data[(self.head + self.size - 1) % len(self.data)] if self.size else -1
        )


q = MyCircularQueue(3)
assert [q.en_queue(v) for v in (1, 2, 3, 4)] == [True, True, True, False]
assert q.rear() == 3 and q.de_queue() and q.en_queue(4) and q.rear() == 4
assert q.front() == 2
print(q.data)

[4, 2, 3]


### 5.1.4 [284. Peeking Iterator](https://leetcode.com/problems/peeking-iterator/) — ★ · Medium

**Problem.** Wrap an iterator to support `peek()` in addition to `next()` and `has_next()`.  
**Example.** over `[1, 2, 3]`: `next()` → `1`, `peek()` → `2`, `next()` → `2`, `next()` → `3`, `has_next()` → `False`  
**Spot it.** Cache one element ahead.

**Ask first**
- Can `peek` be called many times in a row? → Yes — it must not advance.

> 💡 **Idea.** Cache the next element: `peek` reads the cache, `next` returns it and refills from the underlying iterator.

**Brute force:** copy everything into a list first — O(n) memory.

**Steps**
1. Pre-fetch the first element (or a sentinel at the end).
2. `peek` returns the buffer.
3. `next` returns the buffer and pre-fetches again.
4. `has_next` checks for the sentinel.

**Cost:** Time O(1) per operation · Space O(1)

**Likely follow-ups**
- **Generic for any type?** Keep the cache as `Optional[T]` plus a `has_next` flag (don't use `None` as "empty" if `None` is a valid value).
- **Peek k ahead?** Keep a deque buffer of up to k items.

In [38]:
class PeekingIterator:
    """Iterator with one-element lookahead."""

    # unique marker for "no more elements" (None could be a real value)
    _END = object()

    def __init__(self, iterator):
        self.it = iterator
        self._next = next(self.it, self._END)

    def peek(self):
        return self._next

    def next(self):
        value = self._next
        # fetch the following element ahead of time
        self._next = next(self.it, self._END)
        return value

    def has_next(self):
        return self._next is not self._END


pi = PeekingIterator(iter([1, 2, 3]))
assert pi.next() == 1 and pi.peek() == 2 and pi.next() == 2 and pi.next() == 3
assert not pi.has_next()
print("ok")

ok


### 5.1.5 [341. Flatten Nested List Iterator](https://leetcode.com/problems/flatten-nested-list-iterator/) — ★ · Medium

**Problem.** Iterate over the integers of an arbitrarily nested list.  
**Example.** `[[1, 1], 2, [1, 1]]` → `1, 1, 2, 1, 1`  
**Spot it.** Stack of iterators (lazy) — or flatten eagerly with recursion.

**Ask first**
- Can lists be empty (`[[]]`)? → Yes — `has_next` must skip them.

> 💡 **Idea.** Keep a stack of iterators. `has_next` digs in: while the top item is a list, push its iterator; stop when an integer is on top. `next` returns it.

**Brute force:** flatten everything up front into a list — O(n) memory, not lazy.

**Steps**
1. Stack starts with an iterator over the top list.
2. `has_next`: pull from the top iterator; exhausted → pop; a list → push its iterator; an integer → buffer it.
3. `next`: ensure the buffer is filled, then return it.

**Cost:** Time Amortised O(1) per element · Space O(depth)

**Likely follow-ups**
- **Fully lazy / O(depth) memory?** The stack of iterators approach above.
- **Weighted sum by depth (LC 339)?** DFS passing the depth.
- **Python generator version?** A recursive generator with `yield from`.

In [39]:
class NestedIterator:
    """Lazy flattening iterator."""

    def __init__(self, nested):
        # one iterator per open list level
        self.stack = [iter(nested)]
        self.buffer = []

    def has_next(self):
        # advance until an integer is found or nothing is left
        while not self.buffer and self.stack:
            item = next(self.stack[-1], None)
            # this level is exhausted
            if item is None:
                self.stack.pop()
            # descend into a nested list
            elif isinstance(item, list):
                self.stack.append(iter(item))
            else:
                self.buffer.append(item)
        return bool(self.buffer)

    def next(self):
        self.has_next()
        return self.buffer.pop()


it = NestedIterator([[1, 1], 2, [1, [4, [6]]], []])
out = []
while it.has_next():
    out.append(it.next())
assert out == [1, 1, 2, 1, 4, 6]
print(out)

[1, 1, 2, 1, 4, 6]


### 5.1.6 [1396. Design Underground System](https://leetcode.com/problems/design-underground-system/) — ★ · Medium

**Problem.** Check-in/check-out events; `get_average_time(start, end)` over completed trips.  
**Example.** `check_in(1, "A", 3)`, `check_out(1, "B", 8)`, `get_average_time("A", "B")` → `5.0`  
**Spot it.** Two hash maps: open trips by customer, and totals by `(start, end)`.

**Ask first**
- Is every check-out matched by a check-in? → Yes, guaranteed.

> 💡 **Idea.** Map id → `(start station, time)` on check-in. On check-out, add the trip time to a `(start, end)` → `(total, count)` map. Average = total / count.

**Brute force:** store every trip and scan them on each query.

**Steps**
1. Check-in: `open[id] = (station, time)`.
2. Check-out: pop it, add the duration to `totals[(start, end)]` and increment the count.
3. Average = sum / count.

**Cost:** Time O(1) per operation · Space O(customers + routes)

**Likely follow-ups**
- **Median trip time instead of average?** Keep two heaps per route (like LC 295).
- **Only trips in the last hour?** Per-route queue of trips; drop old ones on query (or time buckets).
- **Lots of routes, memory tight?** Store only `(total, count)` per route — already O(routes).

In [40]:
class UndergroundSystem:
    """Average trip times between stations."""

    def __init__(self):
        # customer -> (station, time) of the trip in progress
        self.open = {}
        # (start, end) -> [total time, trip count]
        self.totals = defaultdict(lambda: [0, 0])

    def check_in(self, cid, station, t):
        self.open[cid] = (station, t)

    def check_out(self, cid, station, t):
        start, t0 = self.open.pop(cid)
        # fold the finished trip into its route's totals
        total = self.totals[(start, station)]
        total[0] += t - t0
        total[1] += 1

    def get_average_time(self, start, end):
        total, count = self.totals[(start, end)]
        return total / count


u = UndergroundSystem()
u.check_in(45, "Leyton", 3)
u.check_in(32, "Paradise", 8)
u.check_out(45, "Waterloo", 15)
u.check_out(32, "Cambridge", 22)
assert u.get_average_time("Paradise", "Cambridge") == 14.0
u.check_in(10, "Leyton", 24)
u.check_out(10, "Waterloo", 38)
assert u.get_average_time("Leyton", "Waterloo") == 13.0
print(u.get_average_time("Leyton", "Waterloo"))

13.0


---
# Summary

| Pattern | Recognise it by | Core move | Cost |
|---|---|---|---|
| Farthest reach | jump games | track max reachable index / BFS window | O(n) |
| Restart on failure | gas station, Kadane-like scans | a failed prefix rules out every start inside it | O(n) |
| Last occurrence | partition labels | stretch the window to the last index of every letter | O(n) |
| Range of states | wildcard parentheses | track `[lo, hi]` of possible counts | O(n) |
| Two-pass constraints | candy | fix left neighbours, then right | O(n) |
| Sort by key + greedy | two cities, queue by height, cookies | exchange argument picks the order | O(n log n) |
| Merge intervals | merge / insert | sort by start, extend the last | O(n log n) |
| Activity selection | removals, arrows | sort by end, keep earliest finishers | O(n log n) |
| Active count | meeting rooms, query intervals | heap of ends | O(n log n) |
| Digit math | reverse, palindrome number, multiply strings | `divmod(x, 10)`, carry arrays | O(digits) |
| Fast power | pow | repeated squaring | O(log n) |
| XOR tricks | single / missing numbers | pairs cancel | O(n) |
| Bit DP | counting bits | `bits[i >> 1] + (i & 1)` | O(n) |
| Design | O(1) operations | combine map + array / deque / ring buffer | O(1) |